# Molecular Property Prediction on QM9 with NNConv

Molecular Property Prediction on QM9: Continuous edge-conditioned convolutions predicting quantum chemical properties. This notebook implements the approach with `NNConv` inside a `K3QM9Net` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `NNConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers

title = "Molecular Property Prediction on QM9 with NNConv"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. NNConv Molecular Model
class K3QM9Net(keras.Model):
    def __init__(self, in_channels, edge_dim, out_channels=1):
        super().__init__()
        self.lin0 = layers.Dense(64)
        nn1 = keras.Sequential([layers.Dense(128, activation="relu"), layers.Dense(64 * 64)])
        self.conv1 = k3_layers.NNConv(64, 64, nn1)
        self.lin1 = layers.Dense(32, activation="relu")
        self.lin2 = layers.Dense(out_channels)

    def call(self, x, edge_index, edge_attr, batch=None):
        out = ops.relu(self.lin0(x))
        h = ops.relu(self.conv1(out, edge_index, edge_attr))
        out = k3_layers.global_add_pool(h, batch)
        out = self.lin1(out)
        return self.lin2(out)

k3_model = K3QM9Net(in_channels=11, edge_dim=4, out_channels=1)

# 2. Forward pass test
num_nodes = 20
dummy_x = keras.random.normal((num_nodes, 11))
dummy_edges = ops.convert_to_tensor([[0, 1, 2], [1, 2, 0]], dtype="int64")
dummy_edge_attr = keras.random.normal((3, 4))
dummy_batch = ops.zeros((num_nodes,), dtype="int64")

out = k3_model(dummy_x, dummy_edges, dummy_edge_attr, dummy_batch)
print(f"QM9 NNConv forward pass output shape: {out.shape}")

print("\n✓ K3-Node QM9 NNConv execution completed successfully!")